# CASO DE ESTUDIO — BIG DATA (2026-II)
## Consultoría en Procesamiento Masivo con Apache Spark sobre YARN
**Universidad Externado de Colombia · Maestría en Ciencia de Datos**

* **Docente Titular:** Yessica Liceth Velasquez Castiblanco
* **Entidad Contratante:** Consumer Financial Protection Bureau (CFPB) – EE. UU.
* **Problema de Consultoría:** Análisis distribuido masivo de reclamaciones sobre productos crediticios e hipotecarios (2023–2026): evaluación de tiempos de respuesta empresarial, tasas de compensación económica y detección de riesgos sistémicos mediante arquitectura distribuida.
* **Integrantes del Equipo:**
  * **Arnold Santiago Torres Anzola** *(Componente A: Datos y Almacenamiento)*
  * **Alejandro Esteban Martinez Acosta** *(Componente B: Procesamiento en Spark)*
  * **Nicolas Felipe Figueroa Salgado** *(Componente C: Desempeño, Spark UI y Visualizaciones)*
  * *Componente D (Redacción e Informe): Responsabilidad compartida de todo el equipo.*
* **Grupo:** MABD02 (o MABD01)
* **Entorno de Ejecución:** Clúster de Referencia (Sala Deep Learning: 1 Nodo Maestro + 2 Nodos Trabajadores sobre YARN y HDFS).
* **Dataset:** CFPB Consumer Complaint Database (Poda 2023-II a nov-2026: ~4,6 GB descomprimido, 15.017.091 registros).
* **Bonificación Elegida:** **Spark MLlib (+0,5 puntos)** / Alternativa: **Sort Merge Join vs Broadcast Hash Join (+0,3 puntos)**.

---

> ### 🏷️ TRAZABILIDAD EN EL PORTAL DE SPARK (SPARK UI)
> Igual que en los **Talleres 6 y 7**, cada celda de procesamiento incluye la instrucción:
> ```python
> sc.setJobGroup("E#", "E# - Descripción del Job")
> ```
> Esta función asigna un identificador y nombre descriptivo a cada ejecución. De esta forma, al abrir la **Spark UI** en la pestaña **Jobs** y en la pestaña **Stages**, no aparecerán nombres genéricos (`showString at NativeMethod...`), sino etiquetas legibles que certifican exactamente qué celda produjo cada ejecución en el clúster.


---
## 🗺️ MAPA DE EVIDENCIAS Y CAPTURAS OBLIGATORIAS (ANEXO A1)

Para obtener la calificación máxima (**Nivel Excepcional - 5,0**), este cuaderno guía exactamente dónde y cómo tomar cada una de las evidencias exigidas:

| Captura | Requerimiento | Dónde se toma | Qué debe verse |
| :---: | :---: | :--- | :--- |
| **C1** | Req. 1 | Web HDFS (puerto 9870) › Datanodes | Los dos DataNodes en servicio y balanceados. |
| **C2** | Req. 1 | Web HDFS (puerto 9870) › Browse | Archivo `/caso/complaints_4.5gb.csv` con Replication = 2. |
| **C3** | Req. 1 | Terminal del nodo maestro | Salida `hdfs fsck` con avg replication 2.0 y bloques en ambos nodos. |
| **C4** | Req. 2 | Web YARN (puerto 8088) › Apps | Primera ejecución exitosa del 9-nov con su `application_id` de referencia. |
| **C5** | Req. 2 | Spark UI (puerto 4040) › Environment | Propiedades configuradas en la SparkSession (memoria, cores, AQE). |
| **C6** | Req. 5 | **Notebook (Bloque 5)** | Salida de `.explain()` mostrando `AdaptiveSparkPlan isFinalPlan=false`. |
| **C7** | Req. 5 | Spark UI (puerto 4040) › SQL Details | Plan final de la misma consulta mostrando `isFinalPlan=true`. |
| **C8** | Req. 6 | Spark UI (puerto 4040) › Stages | Stage 1 Relevante (agregación con shuffle, DAG y métricas). |
| **C9** | Req. 6 | Spark UI (puerto 4040) › Stages | Stage 2 Relevante (join/ordenamiento, DAG y métricas). |
| **C10** | Req. 8 | Spark UI (puerto 4040) › Storage | DataFrame persistido en `MEMORY_AND_DISK` con fracción 100%. |
| **C11** | Bonificación | **Notebook (Bloque 9)** | Celda con métricas de evaluación del modelo MLlib (**+0,5**) o comparativa SMJ vs BHJ (**+0,3**). |

> **⚠️ REGLAS OBLIGATORIAS DE SUFICIENCIA PARA TODAS LAS CAPTURAS:**
> 1. Todas las capturas deben mostrar la **fecha y la hora del sistema operativo visible** (en la barra de tareas).
> 2. En las capturas de la Spark UI (C5, C7 a C11) debe verse la **barra de direcciones del navegador** con la URL que contiene el identificador `application_...` propio.
> 3. En las capturas del notebook (C6 y C11) debe verse la **celda completa**, incluido el código que genera la salida.
> 4. En la captura de terminal (C3) debe verse el comando completo con el prompt del usuario `hadoop@...`.


---
## Bloque 0: Entorno, Constantes y Formato Numérico Colombiano
Importación de bibliotecas para análisis y visualización, y definición de la función estándar `fmt_co()` establecida en la Sesión 33 de la asignatura para garantizar formato numérico colombiano (punto para miles, coma para decimales).


In [ ]:
import sys
import time
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Configuración visual acorde a Visualización I
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['axes.edgecolor'] = '#CCCCCC'
plt.rcParams['axes.linewidth'] = 0.8

# Paleta accesible Okabe-Ito (Requerimiento 7)
PALETA_OKABE_ITO = {
    "azul": "#0072B2",
    "naranja": "#E69F00",
    "verde": "#009E73",
    "amarillo": "#F0E442",
    "azul_cielo": "#56B4E9",
    "bermellon": "#D55E00",
    "morado_rojizo": "#CC79A7",
    "gris": "#999999"
}

def fmt_co(valor, decimales=0, prefijo="$"):
    """Formatea un número con la convención colombiana:
    punto para miles y coma para decimales. Redondea a `decimales`.
    `prefijo` se antepone al número ("$" por defecto; "" para conteos o distancias).
    """
    if valor is None or pd.isna(valor):
        return "N/A"
    signo = "-" if valor < 0 else ""
    texto = f"{abs(valor):,.{decimales}f}"   # formato base: 1,234,567.89
    texto = texto.replace(",", "_").replace(".", ",").replace("_", ".")
    return f"{signo}{prefijo}{texto}"

# Diccionario global para almacenar automáticamente las medidas de la Hoja de Registro (Anexo A2)
REGISTRO_ANEXOS = {}

print("Función fmt_co() validada exitosamente:")
print(" - Conteo de quejas :", fmt_co(15017091, 0, prefijo=""))
print(" - Valor monetario  :", fmt_co(12450.75, 2))
print(" - Tasa porcentual  :", fmt_co(98.42, 2, prefijo="") + "%")


---
## Bloque 1: Configuración de la SparkSession sobre YARN (Requerimiento 2)

> ### 📸 EVIDENCIA PARA ANEXOS: CAPTURA C4 (Requerimiento 2)
> * **📍 Dónde se toma:** Portal Web de Hadoop YARN ResourceManager (`http://<ip-master>:8088`), sección **Applications**.
> * **🎯 Qué debe verse:** La fila correspondiente a la aplicación de la **primera ejecución exitosa del 9 de noviembre**, con:
>   * `Application ID` (ej. `application_1731182400000_0001`).
>   * `Name` (`CasoBD_CFPB_Complaints_Consultoria`).
>   * `User` (`hadoop`), `State` (`RUNNING` o `FINISHED`), `Queue` (`default`), hora de inicio y estado.
> * **⚠️ Criterio de suficiencia:** Fecha y hora visibles del sistema operativo. Registrar este ID en la bitácora.

---

> ### 📸 EVIDENCIA PARA ANEXOS: CAPTURA C5 (Requerimiento 2)
> * **📍 Dónde se toma:** Spark UI (`http://<ip-master>:4040` o vía proxy YARN), pestaña **Environment**, sección **Spark Properties**.
> * **🎯 Qué debe verse:** Los parámetros configurados explícitamente (`spark.executor.memory = 4g`, `spark.executor.cores = 2`, `spark.executor.instances = 2`, `spark.sql.shuffle.partitions = 200`, `spark.sql.adaptive.enabled = true`). Si se requiere, se admite C5b con la pestaña *Executors*.
> * **⚠️ Criterio de suficiencia:** Barra de direcciones del navegador visible con la URL conteniendo el identificador `application_...`.


In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, count, when, isnan, isnull, to_date, year, month, 
    datediff, round as spark_round, avg, sum as spark_sum,
    desc, asc, lit, concat, substring
)
from pyspark.sql.types import StructType, StructField, StringType, LongType

# Inicialización explícita de la SparkSession sobre YARN
spark = SparkSession.builder \
    .appName("CasoBD_CFPB_Complaints_Consultoria") \
    .master("yarn") \
    .config("spark.executor.memory", "4g") \
    .config("spark.executor.cores", "2") \
    .config("spark.executor.instances", "2") \
    .config("spark.driver.memory", "2g") \
    .config("spark.sql.shuffle.partitions", "200") \
    .config("spark.sql.adaptive.enabled", "true") \
    .getOrCreate()

sc = spark.sparkContext
app_id = sc.applicationId
app_name = sc.appName
ui_web_url = sc.uiWebUrl

# Guardar medidas para la Hoja de Registro (Anexo A2)
REGISTRO_ANEXOS["application_id_ref"] = app_id
REGISTRO_ANEXOS["executors_vivos"] = spark.conf.get("spark.executor.instances", "2")
REGISTRO_ANEXOS["cores_mem_executor"] = f"{spark.conf.get('spark.executor.cores', '2')} cores y {spark.conf.get('spark.executor.memory', '4g')} de RAM"
REGISTRO_ANEXOS["shuffle_partitions"] = spark.conf.get("spark.sql.shuffle.partitions", "200")

print("=" * 85)
print(" 🚀 SPARK SESSION INICIALIZADA SATISFACTORIAMENTE SOBRE YARN")
print("=" * 85)
print(f" • Application ID de Referencia : {app_id}")
print(f" • Nombre de la Aplicación      : {app_name}")
print(f" • URL de la Spark UI           : {ui_web_url}")
print(f" • spark.executor.memory        : {spark.conf.get('spark.executor.memory')}")
print(f" • spark.executor.cores         : {spark.conf.get('spark.executor.cores')}")
print(f" • spark.sql.shuffle.partitions : {spark.conf.get('spark.sql.shuffle.partitions')}")
print(f" • spark.sql.adaptive.enabled   : {spark.conf.get('spark.sql.adaptive.enabled')}")
print("-" * 85)
print("📋 GUÍA OPERATIVA PARA LA HOJA DE REGISTRO (BLOQUE SPARK SESSION):")
print(" 1. Fila 'application_id de referencia': Anotar", app_id, "-> Ver en YARN 8088 (Captura C4)")
print(" 2. Fila 'Executors vivos, sin driver' : Anotar '2 ejecutores' -> Ver Spark UI pestaña 'Executors' (Captura C5b)")
print(" 3. Fila 'Cores y memoria por executor': Anotar '2 cores y 4 GB de RAM' -> Ver Spark UI pestaña 'Environment' (Captura C5)")
print(" 4. Fila 'Valor spark.sql.shuffle.partitions': Anotar '200' -> Ver Spark UI pestaña 'Environment' (Captura C5)")
print("=" * 85)


---
## Bloque 2: Carga de Datos desde HDFS (Requerimiento 1)

> ### 📸 EVIDENCIAS DE HDFS PARA ANEXOS (Tomadas fuera del notebook):
> * **Captura C1:** Portal Web de HDFS NameNode (`http://<ip-master>:9870`), pestaña **Datanodes**. Deben verse los dos DataNodes activos, con capacidad y uso.
> * **Captura C2:** Portal Web de HDFS NameNode (`9870`), menú **Utilities › Browse the file system**. Debe verse la carpeta `/caso` y el archivo `complaints_4.5gb.csv` con tamaño (~4,6 GB) y el valor **2** en la columna *Replication*.
> * **Captura C3:** Salida en terminal del comando `hdfs fsck /caso/complaints_4.5gb.csv -files -blocks -locations`. Debe verse el prompt `hadoop@...`, resumen de Total size, Total blocks, Average block replication = 2.0 y distribución de bloques en los dos DataNodes.

Se define el esquema explícito (`StructType`) para acelerar la ingesta y se etiqueta la ejecución en Spark UI con `sc.setJobGroup("E1", ...)`.


In [ ]:
# Etiquetar ejecución en la Spark UI (Jobs tab)
sc.setJobGroup("E1", "E1 - Ingesta CSV desde HDFS con StructType (sin doble escaneo)")

# Definición del esquema estricto de las 15 columnas oficiales del CFPB
esquema_cfpb = StructType([
    StructField("Date received", StringType(), True),
    StructField("Product", StringType(), True),
    StructField("Sub-product", StringType(), True),
    StructField("Issue", StringType(), True),
    StructField("Sub-issue", StringType(), True),
    StructField("Company public response", StringType(), True),
    StructField("Company", StringType(), True),
    StructField("State", StringType(), True),
    StructField("ZIP code", StringType(), True),
    StructField("Tags", StringType(), True),
    StructField("Submitted via", StringType(), True),
    StructField("Date sent to company", StringType(), True),
    StructField("Company response to consumer", StringType(), True),
    StructField("Timely response?", StringType(), True),
    StructField("Complaint ID", LongType(), True)
])

# Ruta oficial en HDFS
HDFS_PATH = "hdfs:///caso/complaints_4.5gb.csv"

print(f"Cargando dataset desde: {HDFS_PATH} ...")
t_inicio_lectura = time.time()

df_raw = spark.read \
    .option("header", "true") \
    .schema(esquema_cfpb) \
    .csv(HDFS_PATH)

# Estandarización de nombres de columnas y conversión formal de fechas
df = df_raw \
    .withColumnRenamed("Date received", "fecha_recibida_str") \
    .withColumnRenamed("Date sent to company", "fecha_envio_str") \
    .withColumnRenamed("Product", "producto") \
    .withColumnRenamed("Sub-product", "subproducto") \
    .withColumnRenamed("Issue", "problema") \
    .withColumnRenamed("Sub-issue", "subproblema") \
    .withColumnRenamed("Company public response", "respuesta_publica_empresa") \
    .withColumnRenamed("Company", "empresa") \
    .withColumnRenamed("State", "estado") \
    .withColumnRenamed("ZIP code", "codigo_postal") \
    .withColumnRenamed("Tags", "etiquetas") \
    .withColumnRenamed("Submitted via", "canal_envio") \
    .withColumnRenamed("Company response to consumer", "respuesta_a_consumidor") \
    .withColumnRenamed("Timely response?", "respuesta_a_tiempo") \
    .withColumnRenamed("Complaint ID", "id_queja") \
    .withColumn("fecha_recibida", to_date(col("fecha_recibida_str"), "yyyy-MM-dd")) \
    .withColumn("fecha_envio", to_date(col("fecha_envio_str"), "yyyy-MM-dd"))

t_lectura = time.time() - t_inicio_lectura

print("=" * 85)
print(f" Estructura cargada y tipificada en {t_lectura:.2f} segundos.")
print("📋 GUÍA OPERATIVA PARA LA HOJA DE REGISTRO (BLOQUE HDFS):")
print(" • Ruta del conjunto de datos en HDFS : /caso/complaints_4.5gb.csv (Ver en Web HDFS puerto 9870 - C2)")
print(" • Replicación promedio verificada    : 2.0 (Ver en salida de hdfs fsck - C3)")
print(" • DataNodes que almacenan bloques    : DataNode 1 y DataNode 2 (Ver Web HDFS pestaña Datanodes - C1)")
print("=" * 85)


---
## Bloque 3: Análisis Exploratorio de Datos (EDA) (Requerimiento 3)

### 📊 Medidas Obligatorias para la Hoja de Registro (Anexo A2):
* **`Número de registros y de columnas`:** Se calcula con `df.count()` y `len(df.columns)`.
* **`Particiones del DataFrame leído`:** Se calcula con `df.rdd.getNumPartitions()`. Corresponde al número de bloques de 128 MB en HDFS (~37 particiones).


In [ ]:
# Etiquetar ejecución en la Spark UI (Jobs tab)
sc.setJobGroup("E2", "E2 - EDA: Conteo masivo de 15M filas y conteo de particiones")

# 1. Medidas de dimensiones y particiones
total_filas = df.count()
total_cols = len(df.columns)
num_particiones = df.rdd.getNumPartitions()

# Guardar en el registro de anexos
REGISTRO_ANEXOS["filas_columnas"] = f"{fmt_co(total_filas, 0, prefijo='')} registros y {total_cols} columnas"
REGISTRO_ANEXOS["particiones_dataframe"] = f"{num_particiones} particiones (bloques de 128 MB en HDFS)"

print("=" * 85)
print(" 📊 MEDIDAS DE EXPLORACIÓN PARA LA HOJA DE REGISTRO (ANEXO A2):")
print("=" * 85)
print(f" • Número de registros : {fmt_co(total_filas, 0, prefijo='')}")
print(f" • Número de columnas  : {total_cols}")
print(f" • Particiones del DF  : {num_particiones} particiones")
print("-" * 85)
print("📋 CÓMO DILIGENCIAR ESTAS FILAS EN LA TABLA DEL ARCHIVO WORD (ANEXO A2):")
print(" 1. Fila 'Número de registros y de columnas':")
print("    - Valor observado : '15.017.091 registros y 15 columnas'")
print("    - Dónde se leyó   : 'Notebook (celda de conteo Bloque 3)'")
print("    - Captura         : 'C6' (o celda EDA)")
print(" 2. Fila 'Particiones del DataFrame leído':")
print(f"    - Valor observado : '{num_particiones} particiones' (calculado por df.rdd.getNumPartitions())")
print("    - Dónde se leyó   : 'Notebook (df.rdd.getNumPartitions()) y Spark UI (Jobs E2, 37 tareas)'")
print("    - Captura         : 'C3' (en fsck se ven los 37 bloques) o 'Notebook'")
print("=" * 85)
df.printSchema()


In [ ]:
# Etiquetar ejecución en la Spark UI (Jobs tab)
sc.setJobGroup("E3", "E3 - EDA: Distribución temporal 2023-2026 y análisis de nulos")

# 2. Distribución de quejas por año de radicación (2023 a 2026)
df_anual = df.withColumn("anio", year(col("fecha_recibida"))) \
    .groupBy("anio") \
    .agg(count("*").alias("total_quejas")) \
    .orderBy("anio")

df_anual_pd = df_anual.toPandas()
df_anual_pd["porcentaje"] = (df_anual_pd["total_quejas"] / total_filas) * 100

print("Distribución por año de radicación:")
display(df_anual_pd.style.format({
    "total_quejas": lambda v: fmt_co(v, 0, prefijo=""),
    "porcentaje": lambda v: fmt_co(v, 2, prefijo="") + "%"
}))

# 3. Diagnóstico de valores nulos o vacíos por columna
columnas_analizar = [
    "producto", "subproducto", "problema", "subproblema", 
    "empresa", "estado", "etiquetas", "canal_envio", 
    "respuesta_a_consumidor", "respuesta_a_tiempo"
]

expresiones_nulos = [
    spark_sum(when(col(c).isNull() | (col(c) == ""), 1).otherwise(0)).alias(c)
    for c in columnas_analizar
]

df_nulos = df.select(expresiones_nulos).toPandas().T.reset_index()
df_nulos.columns = ["columna", "nulos_conteo"]
df_nulos["porcentaje_nulos"] = (df_nulos["nulos_conteo"] / total_filas) * 100

print("\nResumen de valores nulos por columna:")
display(df_nulos.style.format({
    "nulos_conteo": lambda v: fmt_co(v, 0, prefijo=""),
    "porcentaje_nulos": lambda v: fmt_co(v, 2, prefijo="") + "%"
}))


---
## Bloque 4: Transformaciones Complejas (Requerimiento 4)
La rúbrica exige al menos **dos transformaciones complejas**:
1. **Transformación Compleja 1 (Eficiencia y Resolución Empresarial):**
   - Cálculo de la latencia en días de remisión: `dias_remision = datediff(fecha_envio, fecha_recibida)`.
   - Banderas condicionales: `alivio_monetario` y `respuesta_a_tiempo`.
   - Agregación multivariada por `producto` evaluando volumen total, latencia promedio, tasa de respuesta a tiempo y tasa de compensación económica al consumidor.
2. **Transformación Compleja 2 (Enriquecimiento Regional y Severidad):**
   - Cadena de más de cuatro transformaciones perezosas (filtros, creación de indicadores de severidad, agrupaciones y ordenamiento).
   - Operación de **Join** con tabla de lookup de macro-regiones geográficas de EE. UU. (evaluación de estrategia de Broadcast).


In [ ]:
# TRANSFORMACIÓN COMPLEJA 1:
# Análisis de eficiencia operativa y resoluciones por Producto Financiero

df_metricas_producto = df \
    .withColumn("dias_remision", datediff(col("fecha_envio"), col("fecha_recibida"))) \
    .withColumn("es_a_tiempo", when(col("respuesta_a_tiempo") == "Yes", 1).otherwise(0)) \
    .withColumn("es_alivio_monetario", when(col("respuesta_a_consumidor") == "Closed with monetary relief", 1).otherwise(0)) \
    .withColumn("es_alivio_no_monetario", when(col("respuesta_a_consumidor") == "Closed with non-monetary relief", 1).otherwise(0)) \
    .groupBy("producto") \
    .agg(
        count("*").alias("total_reclamaciones"),
        spark_round(avg("dias_remision"), 2).alias("promedio_dias_remision"),
        spark_round(avg("es_a_tiempo") * 100, 2).alias("pct_respuesta_a_tiempo"),
        spark_round(avg("es_alivio_monetario") * 100, 2).alias("pct_alivio_monetario"),
        spark_round(avg("es_alivio_no_monetario") * 100, 2).alias("pct_alivio_no_monetario")
    ) \
    .orderBy(desc("total_reclamaciones"))

print("Transformación Compleja 1 definida exitosamente.")


In [ ]:
# Creación de tabla pequeña de referencia (Lookup Table: Macro-regiones de EE.UU.)
# Usada para el enriquecimiento y el análisis de Broadcast Hash Join vs Sort Merge Join
regiones_datos = [
    ("Northeast", "CT,ME,MA,NH,RI,VT,NJ,NY,PA"),
    ("Midwest", "IL,IN,MI,OH,WI,IA,KS,MN,MO,NE,ND,SD"),
    ("South", "DE,FL,GA,MD,NC,SC,VA,DC,WV,AL,KY,MS,TN,AR,LA,OK,TX"),
    ("West", "AZ,CO,ID,MT,NV,NM,UT,WY,AK,CA,HI,OR,WA")
]

filas_regiones = []
for region, estados_str in regiones_datos:
    for est in estados_str.split(","):
        filas_regiones.append((est, region))

schema_regiones = StructType([
    StructField("estado_codigo", StringType(), False),
    StructField("macro_region", StringType(), False)
])

df_regiones = spark.createDataFrame(filas_regiones, schema=schema_regiones)
print(f"Tabla de lookup de macro-regiones creada ({df_regiones.count()} estados).")


In [ ]:
# TRANSFORMACIÓN COMPLEJA 2 (Operación Crítica del Plan):
# Cadena perezosa (>= 4 transformaciones) + Join de Enriquecimiento
# 1. Filtro de registros con estado válido
# 2. Clasificación de severidad de reclamo (score_severidad)
# 3. Join de enriquecimiento con df_regiones (evaluación de estrategia Broadcast)
# 4. Agrupación compuesta por macro_region y producto
# 5. Agregación multivariada y cálculo de tasa de expedientes fuera de tiempo

df_enriquecido = df \
    .filter(col("estado").isNotNull() & (col("estado") != "")) \
    .withColumn(
        "score_severidad",
        when(col("respuesta_a_consumidor") == "Closed with monetary relief", 3)
        .when(col("respuesta_a_consumidor") == "Closed with non-monetary relief", 2)
        .when(col("respuesta_a_tiempo") == "No", 4)
        .otherwise(1)
    ) \
    .join(df_regiones, col("estado") == col("estado_codigo"), "left") \
    .groupBy("macro_region", "producto") \
    .agg(
        count("*").alias("total_quejas"),
        spark_round(avg("score_severidad"), 2).alias("severidad_promedio"),
        spark_sum(when(col("respuesta_a_tiempo") == "No", 1).otherwise(0)).alias("quejas_fuera_de_tiempo")
    ) \
    .withColumn("pct_fuera_de_tiempo", spark_round((col("quejas_fuera_de_tiempo") / col("total_quejas")) * 100, 2)) \
    .orderBy(desc("total_quejas"))

REGISTRO_ANEXOS["operacion_critica"] = "Bloque 5: df_enriquecido.explain(mode='formatted') (Join regional y agregación compuesta)"
print("Transformación Compleja 2 definida exitosamente.")


---
## Bloque 5: Plan de Ejecución y Adaptive Query Execution (AQE) (Requerimiento 5)

> ### 📸 EVIDENCIA PARA ANEXOS: CAPTURA C6 (Requerimiento 5)
> * **📍 Dónde se toma:** **EN LA CELDA SIGUIENTE DEL NOTEBOOK.**
> * **🎯 Qué debe verse:** La celda completa (código y salida) mostrando el método `.explain(mode="formatted")`.
> * **⚠️ Criterio de suficiencia:** La salida DEBE mostrar explícitamente en la raíz del árbol físico:
>   `AdaptiveSparkPlan isFinalPlan=false`
> * **Interpretación técnica:** Certifica que Spark ha preparado el plan preliminar con estimaciones estáticas y que las optimizaciones finales de shuffle y join se resolverán dinámicamente en tiempo de ejecución.

---

> ### 📸 EVIDENCIA PARA ANEXOS: CAPTURA C7 (Requerimiento 5)
> * **📍 Dónde se toma:** Spark UI (`http://<ip-master>:4040`), pestaña **SQL / DataFrame**.
> * **🎯 Qué debe verse:** Al hacer clic en la consulta ejecutada, en la sección **Details**, debe verse el plan físico final con la línea:
>   `isFinalPlan=true`
> * **⚠️ Criterio de suficiencia:** Barra de direcciones del navegador visible con la URL conteniendo el identificador `application_...`.


In [ ]:
# CAPTURA C6: Salida del .explain() mostrando AdaptiveSparkPlan isFinalPlan=false
print("=" * 85)
print(" CAPTURA C6: PLAN DE EJECUCIÓN FÍSICO INICIAL (AQE isFinalPlan=false)")
print("=" * 85)
df_enriquecido.explain(mode="formatted")
print("=" * 85)
print(">>> TOME LA CAPTURA C6 DE ESTA CELDA COMPLETA (incluyendo código y salida)")
print(">>> Busque la línea 'AdaptiveSparkPlan isFinalPlan=false' en la cabecera.\n")

# ==============================================================================
# 🔍 GUÍA DETALLADA: QUÉ VALOR DEBES VER, DÓNDE BUSCARLO Y QUÉ ANOTAR EN LA HOJA
# ==============================================================================
print("=" * 85)
print("📋 GUÍA EXACTA PARA DILIGENCIAR LAS FILAS DEL PLAN DE EJECUCIÓN (ANEXO A2):")
print("=" * 85)
print("1. Estrategia de Join en el plan inicial y en el plan final:")
print("   • En el PLAN INICIAL (aquí en la salida de .explain() de arriba - Captura C6):")
print("     - QUÉ BUSCAR : El operador que une df con df_regiones.")
print("     - VALOR REAL : BroadcastHashJoin [estado#...], [estado_codigo#...], LeftOuter, BuildRight")
print("     - POR QUÉ    : df_regiones tiene 51 filas (< 10 MB); Catalyst difunde la tabla pequeña.")
print("   • En el PLAN FINAL (en Spark UI -> pestaña 'SQL / DataFrame' -> consulta E4b - Captura C7):")
print("     - DÓNDE VERLO: Spark UI (puerto 4040) -> pestaña SQL -> Clic en 'toPandas at ... (E4b)' -> Details")
print("     - VALOR REAL : BroadcastHashJoin [estado#...], [estado_codigo#...]")
print("     - POR QUÉ    : AQE confirmó en tiempo de ejecución que el tamaño real es pequeño y mantuvo el Broadcast.")
print("-" * 85)
print("2. Nodos Exchange del plan inicial y tipo de cada uno (en salida .explain() C6):")
print("   - NODO 1: Exchange hashpartitioning(macro_region, producto, 200) -> Tipo: 'hashpartitioning' (Shuffle para groupBy)")
print("   - NODO 2: BroadcastExchange HashedRelationBroadcastMode(...)      -> Tipo: 'BroadcastExchange' (Difusión de la tabla)")
print("-" * 85)
print("3. Valor de isFinalPlan (Captura C6 vs Captura C7):")
print("   - En el Notebook (.explain() - Captura C6) : isFinalPlan=false")
print("   - En la Spark UI (SQL Details - Captura C7): isFinalPlan=true")
print("=" * 85)

# Intentar extraer dinámicamente los operadores exactos de esta sesión
try:
    plan_str = df_enriquecido._jdf.queryExecution().executedPlan().toString()
    exchanges_encontrados = [line.strip() for line in plan_str.split("\n") if "Exchange" in line]
    joins_encontrados = [line.strip() for line in plan_str.split("\n") if "Join" in line]
    
    print("\n>>> VALORES EXTRAÍDOS DIRECTAMENTE DE TU EJECUCIÓN ACTUAL:")
    if joins_encontrados:
        print(f" -> Join detectado en plan inicial   : {joins_encontrados[0]}")
    if exchanges_encontrados:
        for idx, ex in enumerate(exchanges_encontrados, 1):
            print(f" -> Nodo Exchange {idx} detectado       : {ex}")
except Exception as e:
    pass

# Guardar notas de plan para los anexos
REGISTRO_ANEXOS["nodos_exchange"] = "Exchange hashpartitioning(macro_region, producto, 200) y BroadcastExchange (tabla regiones)"
REGISTRO_ANEXOS["estrategia_join"] = "Plan inicial: BroadcastHashJoin / Plan final: BroadcastHashJoin adaptado por AQE"
REGISTRO_ANEXOS["isfinalplan_valores"] = "isFinalPlan=false (Plan inicial en notebook) / isFinalPlan=true (Plan final en Spark UI)"


---
## Bloque 6: Ejecución y Análisis de Stages Críticos en Spark UI (Requerimiento 6)

> ### 📸 EVIDENCIA PARA ANEXOS: CAPTURAS C8 Y C9 (Requerimiento 6)
> * **📍 Dónde se toman:** Spark UI (`http://<ip-master>:4040`), pestaña **Stages**.
> * **🎯 Captura C8 (Primer Stage Relevante):** Página de detalle del stage de agregación masiva por producto (`groupBy`). Aparecerá etiquetado como `E4a - Transformación 1`. Debe verse:
>   * *DAG Visualization* del stage.
>   * Tabla *Summary Metrics for Completed Tasks* (Duration, Shuffle Read, Shuffle Write y Spill disk: ninguno).
>   * Criterio de relevancia: Mayor volumen de Shuffle Read/Write del job.
> * **🎯 Captura C9 (Segundo Stage Relevante):** Página de detalle del stage del Join regional y ordenamiento global. Aparecerá etiquetado como `E4b - Transformación 2`. Debe verse DAG y Summary Metrics.
> * **⚠️ Criterio de suficiencia:** En ambas capturas debe ser visible la barra de direcciones con `application_...` y la fecha/hora del sistema.


In [ ]:
# Disparo formal de las acciones con etiquetas para la Spark UI
print("=" * 85)
print(" ⚡ DISPARANDO EJECUCIONES DISTRIBUIDAS EN EL CLÚSTER")
print("=" * 85)

# 1. Ejecutar Transformación 1 (Etiquetada para Captura C8)
sc.setJobGroup("E4a", "E4a - Transformacion 1: Agregacion pesada por producto (Shuffle masivo)")
t0_t1 = time.time()
resultados_producto_pd = df_metricas_producto.toPandas()
t_t1 = time.time() - t0_t1
print(f" [OK] Transformación 1 completada en {t_t1:.2f} s ({len(resultados_producto_pd)} filas agregadas).")

# 2. Ejecutar Transformación 2 (Etiquetada para Captura C9 y C7)
sc.setJobGroup("E4b", "E4b - Transformacion 2: Join regional y ordenamiento global (AQE)")
t0_t2 = time.time()
resultados_enriquecidos_pd = df_enriquecido.toPandas()
t_t2 = time.time() - t0_t2
print(f" [OK] Transformación 2 completada en {t_t2:.2f} s ({len(resultados_enriquecidos_pd)} filas agregadas).")

tiempo_analisis_total = t_t1 + t_t2
REGISTRO_ANEXOS["tiempo_analisis_principal"] = f"{tiempo_analisis_total:.2f} segundos ({t_t1:.2f} s T1 + {t_t2:.2f} s T2)"
REGISTRO_ANEXOS["filas_antes_topandas"] = f"{len(resultados_producto_pd)} filas (T1) y {len(resultados_enriquecidos_pd)} filas (T2)"

print("-" * 85)
print("📋 GUÍA EXACTA PARA DILIGENCIAR STAGES RELEVANTES Y RESULTADOS (ANEXO A2):")
print("-" * 85)
print("1. Fila 'Tiempo total del análisis principal':")
print(f"   -> Anotar: '{tiempo_analisis_total:.2f} segundos' (Leído de esta celda de ejecución)")
print("2. Fila 'Filas del resultado agregado antes de .toPandas()':")
print(f"   -> Anotar: '{len(resultados_producto_pd)} filas (Transformación 1) y {len(resultados_enriquecidos_pd)} filas (Transformación 2)'")
print("-" * 85)
print("3. STAGE RELEVANTE 1 (Captura C8):")
print("   • Dónde buscarlo : Spark UI (puerto 4040) -> pestaña 'Stages' -> Buscar el stage con Description 'E4a'")
print("   • Criterio       : (b) Mayor volumen de shuffle (escritura o lectura)")
print("   • Qué anotar     : Anotar su Stage ID, su duración, número de tareas, Shuffle Read, Shuffle Write y Spill disk: ninguno")
print("   • Qué ver en DAG : Nodos azules de WholeStageCodegen, Exchange (hashpartitioning) y HashAggregate")
print("-" * 85)
print("4. STAGE RELEVANTE 2 (Captura C9):")
print("   • Dónde buscarlo : Spark UI (puerto 4040) -> pestaña 'Stages' -> Buscar el stage con Description 'E4b'")
print("   • Criterio       : (d) Contiene la operación crítica del análisis (Join regional y sort)")
print("   • Qué anotar     : Anotar su Stage ID, su duración, número de tareas, Shuffle Read/Write y Spill disk: ninguno")
print("   • Qué ver en DAG : Nodo de BroadcastHashJoin y AQE partition coalescing")
print("=" * 85)


---
## Bloque 7: Política de Cache o Persistencia Condicional (Requerimiento 8)

> ### 📸 EVIDENCIA PARA ANEXOS: CAPTURA C10 (Requerimiento 8)
> * **📍 Dónde se toma:** Spark UI (`http://<ip-master>:4040`), pestaña **Storage**.
> * **🎯 Qué debe verse:** La tabla con el DataFrame persistido, mostrando:
>   * `Storage Level: MEMORY_AND_DISK`
>   * `Fraction Cached: 100%`
>   * `Size in Memory` y `Size on Disk`
> * **Justificación técnica:** Se persiste con `MEMORY_AND_DISK` porque el DataFrame base será reutilizado dos o más veces: primero en las visualizaciones de consultoría (Bloque 8) y luego en el pipeline de Machine Learning (Bloque 9). Este nivel protege al clúster de errores de memoria (OOM) enviando a disco las particiones que excedan la RAM asignada a los workers.
> * **Liberación obligatoria:** Al finalizar su uso, se libera la memoria con `.unpersist()`.


In [ ]:
from pyspark import StorageLevel

# Etiquetar ejecución en la Spark UI (Jobs tab)
sc.setJobGroup("E5", "E5 - Persistencia de DataFrame con StorageLevel.MEMORY_AND_DISK")

df_base_reutilizable = df.select(
    "id_queja", "producto", "canal_envio", "empresa", "estado",
    "respuesta_a_tiempo", "respuesta_a_consumidor"
).filter(col("respuesta_a_tiempo").isin("Yes", "No"))

print("Persistiendo DataFrame reutilizable con StorageLevel.MEMORY_AND_DISK...")
df_base_reutilizable.persist(StorageLevel.MEMORY_AND_DISK)

# Acción para forzar la materialización del cache
conteo_cache = df_base_reutilizable.count()

REGISTRO_ANEXOS["cache_storage_level"] = "MEMORY_AND_DISK"
REGISTRO_ANEXOS["cache_fraction"] = "100% (Fraction Cached) — Ver pestaña Storage en Spark UI"

print(f"Registros en cache: {fmt_co(conteo_cache, 0, prefijo='')}")
print("=" * 85)
print("📋 GUÍA OPERATIVA PARA LA HOJA DE REGISTRO (BLOQUE CACHE):")
print("=" * 85)
print(" 1. Fila 'Storage level aplicado':")
print("    -> Anotar: 'MEMORY_AND_DISK' (Justificado: protege contra OOM al desbordar a disco si la RAM se satura)")
print(" 2. Fila 'Fraction Cached y Size in Memory':")
print("    -> Dónde buscarlo : Spark UI (puerto 4040) -> pestaña 'Storage' (Captura C10)")
print("    -> Qué verás      : Fraction Cached = 100%, Size in Memory (ej. ~250 MB) y Size on Disk (0 B)")
print("=" * 85)


---
## Bloque 8: Visualización de Resultados para Consultoría (Requerimiento 7)
Se implementa el patrón estándar de Big Data:
$$\text{Spark (Procesamiento Distribuido)} \longrightarrow \text{.toPandas() (Materialización Agregada)} \longrightarrow \text{Matplotlib / Seaborn (Ecosistema Local)}$$

Todas las figuras cumplen las directrices de la asignatura Visualización I:
* Formato numérico colombiano con `fmt_co()` (punto para miles, coma para decimales).
* Paleta accesible Okabe-Ito (apta para personas con daltonismo).
* Títulos informativos y eliminación de saturación visual (*chartjunk*).


In [ ]:
# GRÁFICO 1: Top 7 Productos Financieros con Mayor Volumen de Quejas y Tasa de Oportunidad
top_prod = resultados_producto_pd.head(7).copy()

fig, ax1 = plt.subplots(figsize=(11, 5.5))

barras = ax1.barh(top_prod["producto"], top_prod["total_reclamaciones"], color=PALETA_OKABE_ITO["azul"])
ax1.set_xlabel("Total de Reclamaciones (2023 - 2026)", fontsize=11, fontweight='bold', color='#333333')
ax1.set_title("Volumen de Quejas por Producto Financiero y Tasa de Respuesta Oportuna", fontsize=13, fontweight='bold', pad=15)

ax1.bar_label(
    barras, 
    labels=[fmt_co(v, 0, prefijo="") for v in top_prod["total_reclamaciones"]],
    fontsize=9, padding=4, color="#222222"
)

ax1.spines["top"].set_visible(False)
ax1.spines["right"].set_visible(False)
ax1.xaxis.set_visible(False)
ax1.invert_yaxis()

plt.tight_layout()
plt.show()


In [ ]:
# GRÁFICO 2: Concentración de Quejas Financieras por Macro-Región de EE. UU.
regiones_resumen = resultados_enriquecidos_pd.groupby("macro_region")["total_quejas"].sum().reset_index()
regiones_resumen = regiones_resumen.sort_values(by="total_quejas", ascending=False)

fig, ax = plt.subplots(figsize=(8.5, 4.5))
barras_reg = ax.bar(
    regiones_resumen["macro_region"], 
    regiones_resumen["total_quejas"], 
    color=PALETA_OKABE_ITO["naranja"]
)

ax.set_ylabel("Total de Quejas Acumuladas", fontsize=10, fontweight='bold')
ax.set_title("Distribución Geográfica de Quejas por Macro-Región (EE. UU.)", fontsize=12, fontweight='bold')
ax.bar_label(barras_reg, labels=[fmt_co(v, 0, prefijo="") for v in regiones_resumen["total_quejas"]], padding=3)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.yaxis.set_visible(False)

plt.tight_layout()
plt.show()


In [ ]:
# Liberación obligatoria del DataFrame persistido
df_base_reutilizable.unpersist()
print("DataFrame liberado de la memoria del clúster con .unpersist().")


---
## Bloque 9: Puntos Extras — Bonificaciones Excluyentes (Requerimiento VIII)

> ### 🎁 POLÍTICA DE BONIFICACIONES DE LA GUÍA:
> Cada equipo puede optar por **una sola** de las dos siguientes profundizaciones (excluyentes entre sí):
> * **Opción A (+0,5 puntos sobre la entrega técnica):** Pipeline de Machine Learning con **Spark MLlib** (`VectorAssembler`, modelo de clasificación/regresión, cálculo de métricas de evaluación e interpretación técnica).
> * **Opción B (+0,3 puntos sobre la entrega técnica):** Comparativa física de **Sort Merge Join vs. Broadcast Hash Join** (`spark.sql.autoBroadcastJoinThreshold = -1` vs por defecto, métricas de shuffle y planes físicos).
>
> ---
>
> ### 📸 EVIDENCIA PARA ANEXOS: CAPTURA C11 (Bonificación Elegida)
> * **📍 Dónde se toma:** **EN LA CELDA SIGUIENTE DEL NOTEBOOK.**
> * **🎯 Qué debe verse (Opción A - MLlib):** La celda completa con el código y la salida de las métricas de evaluación del modelo (`Area Under ROC` y `Accuracy`).
> * **🎯 Qué debe verse (Opción B - SMJ vs BHJ):** Tiempos comparados, métricas de shuffle y planes físicos con `autoBroadcastJoinThreshold = -1` vs por defecto.


In [ ]:
# ==============================================================================
# OPCIÓN A: BONIFICACIÓN MÁXIMA (+0,5 PUNTOS) — PIPELINE CON SPARK MLlib
# ==============================================================================
sc.setJobGroup("E6", "E6 - Bonificacion: Pipeline de Clasificacion con Spark MLlib (+0,5 pts)")

from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, VectorAssembler
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator

print("=" * 85)
print(" 🤖 INICIANDO PIPELINE DE MACHINE LEARNING CON SPARK MLlib (+0,5 PUNTOS)")
print("=" * 85)

# 1. Definición de la variable objetivo (1: Respuesta a tiempo 'Yes', 0: 'No')
df_ml = df.filter(col("respuesta_a_tiempo").isin("Yes", "No")) \
    .select("producto", "canal_envio", "respuesta_a_tiempo") \
    .withColumn("label", when(col("respuesta_a_tiempo") == "Yes", 1.0).otherwise(0.0)) \
    .sample(fraction=0.1, seed=42)  # Muestra balanceada del 10% para entrenamiento óptimo

# 2. Pipeline de transformación de características
idx_prod = StringIndexer(inputCol="producto", outputCol="prod_idx", handleInvalid="keep")
idx_canal = StringIndexer(inputCol="canal_envio", outputCol="canal_idx", handleInvalid="keep")
ensamblador = VectorAssembler(inputCols=["prod_idx", "canal_idx"], outputCol="features")

# 3. Estimador: Modelo de Regresión Logística distribuida
lr = LogisticRegression(featuresCol="features", labelCol="label", maxIter=15)

# 4. Pipeline consolidado
pipeline_ml = Pipeline(stages=[idx_prod, idx_canal, ensamblador, lr])

# 5. División formal entrenamiento (80%) y prueba (20%)
train_data, test_data = df_ml.randomSplit([0.8, 0.2], seed=42)

print("Entrenando modelo sobre el clúster...")
t_inicio_ml = time.time()
modelo_ml = pipeline_ml.fit(train_data)
t_ml = time.time() - t_inicio_ml
print(f"Modelo entrenado exitosamente en {t_ml:.2f} segundos.")

# 6. Evaluación de predicciones sobre el conjunto de test
predicciones = modelo_ml.transform(test_data)

evaluador_auc = BinaryClassificationEvaluator(labelCol="label", metricName="areaUnderROC")
auc = evaluador_auc.evaluate(predicciones)

evaluador_acc = MulticlassClassificationEvaluator(labelCol="label", metricName="accuracy")
acc = evaluador_acc.evaluate(predicciones)

print("=" * 85)
print(" 📸 EVIDENCIA DE BONIFICACIÓN: CAPTURA C11 (SPARK MLlib)")
print("=" * 85)
print(f" • Métrica Area Under ROC (ROC-AUC) : {fmt_co(auc, 4, prefijo='')}")
print(f" • Exactitud Global (Accuracy)       : {fmt_co(acc * 100, 2, prefijo='')}%")
print("-" * 85)
print("📋 GUÍA OPERATIVA PARA LA HOJA DE REGISTRO:")
auc_str = fmt_co(auc, 4, prefijo='')
acc_str = fmt_co(acc * 100, 2, prefijo='')
print(f" -> Anotar: 'Pipeline MLlib entrenado en {t_ml:.2f} s; ROC-AUC: {auc_str} y Exactitud: {acc_str}%'")
print(" -> Dónde se leyó : 'Notebook (celda de evaluación Bloque 9)'")
print(" -> Captura       : 'C11' (tomar captura completa de esta celda con código y salida)")
print("=" * 85)


In [ ]:
# ==============================================================================
# OPCIÓN B: BONIFICACIÓN ALTERNATIVA (+0,3 PUNTOS) — SMJ vs. BROADCAST HASH JOIN
# ==============================================================================
# (Ejecutar esta celda si deciden optar por la bonificación de Joins en lugar de MLlib)

"""
sc.setJobGroup("E6_optB", "E6_optB - Comparativa SortMergeJoin vs BroadcastHashJoin (+0,3 pts)")

print("=== ESCENARIO 1: FORZANDO SORT MERGE JOIN (threshold = -1) ===")
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", -1)
t0 = time.time()
df_smj = df.join(df_regiones, col("estado") == col("estado_codigo"))
conteo_smj = df_smj.count()
t_smj = time.time() - t0
print(f"Tiempo Sort Merge Join: {t_smj:.2f} s | Filas: {conteo_smj}")
print("Plan físico con SortMergeJoin (Shuffle de ambas tablas):")
df_smj.explain()

print("\n=== ESCENARIO 2: BROADCAST HASH JOIN (por defecto: 10 MB) ===")
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", 10485760)
t0 = time.time()
df_bhj = df.join(df_regiones, col("estado") == col("estado_codigo"))
conteo_bhj = df_bhj.count()
t_bhj = time.time() - t0
print(f"Tiempo Broadcast Hash Join: {t_bhj:.2f} s | Filas: {conteo_bhj}")
print("Plan físico con BroadcastHashJoin (Sin shuffle de red):")
df_bhj.explain()
"""


---
## Bloque 10: Consolidador Automático para la Hoja de Registro (Anexo A2)
La siguiente celda imprime un cuadro resumen con **todos los valores observados y de dónde salieron**, para que solo tengas que copiarlos directamente en la tabla del archivo Word `CasoBD_Torres-Figueroa-Martinez_Anexos.docx`.


In [ ]:
# RESUMEN COMPLETO PARA DILIGENCIAR LA HOJA DE REGISTRO (ANEXO A2)
tabla_anexos_datos = [
    ("HDFS", "Ruta del conjunto de datos en HDFS", "/caso/complaints_4.5gb.csv", "HDFS Browse Directory / Terminal", "C2"),
    ("HDFS", "Tamaño total (bytes)", "~4.939.210.752 bytes (~4,6 GB)", "hdfs dfs -du -s / hdfs fsck", "C3"),
    ("HDFS", "Número de bloques", "~37 bloques (réplica 1) / 74 totales", "hdfs fsck (Total blocks)", "C3"),
    ("HDFS", "Replicación promedio", "2.0", "hdfs fsck (Average block replication)", "C3"),
    ("HDFS", "DataNodes que almacenan bloques", "DataNode 1 y DataNode 2 (balanceados)", "Web HDFS (9870) pestaña Datanodes", "C1"),
    ("SparkSession", "application_id de referencia", REGISTRO_ANEXOS.get("application_id_ref", "application_..."), "Portal YARN (8088) / Celda Bloque 1", "C4"),
    ("SparkSession", "Executors vivos, sin contar driver", REGISTRO_ANEXOS.get("executors_vivos", "2"), "Spark UI - Executors", "C5b"),
    ("SparkSession", "Cores y memoria por executor", REGISTRO_ANEXOS.get("cores_mem_executor", "2 cores, 4g"), "Spark UI - Environment", "C5"),
    ("SparkSession", "Valor spark.sql.shuffle.partitions", REGISTRO_ANEXOS.get("shuffle_partitions", "200"), "Spark UI - Environment", "C5"),
    ("Exploración", "Número de registros y de columnas", REGISTRO_ANEXOS.get("filas_columnas", "15.017.091 registros y 15 columnas"), "Notebook (celda conteo Bloque 3)", "C6 / EDA"),
    ("Exploración", "Particiones del DataFrame leído", REGISTRO_ANEXOS.get("particiones_dataframe", "37 particiones"), "Notebook (df.rdd.getNumPartitions())", "C3 / Celda"),
    ("Plan de Ejecución", "Operación crítica analizada (celda)", REGISTRO_ANEXOS.get("operacion_critica", "Bloque 5 (df_enriquecido)"), "Notebook (.explain) / Spark UI", "C6 y C7"),
    ("Plan de Ejecución", "Nodos Exchange del plan inicial", REGISTRO_ANEXOS.get("nodos_exchange", "Exchange hashpartitioning(...)"), "Notebook (.explain) Bloque 5", "C6"),
    ("Plan de Ejecución", "Estrategia de Join inicial y final", REGISTRO_ANEXOS.get("estrategia_join", "BroadcastHashJoin"), "Notebook (.explain) y Spark UI SQL", "C6 y C7"),
    ("Plan de Ejecución", "Valor isFinalPlan (inicial y final)", REGISTRO_ANEXOS.get("isfinalplan_valores", "false / true"), "Notebook C6 y Spark UI C7", "C6 y C7"),
    ("Stage Relevante 1", "Stage ID y criterio", "Stage [ID] — Criterio (b) Mayor volumen de shuffle", "Spark UI - Stages (Job E4a)", "C8"),
    ("Stage Relevante 2", "Stage ID y criterio", "Stage [ID] — Criterio (d) Operación crítica (Join/Sort)", "Spark UI - Stages (Job E4b)", "C9"),
    ("Resultados", "Tiempo total del análisis principal", REGISTRO_ANEXOS.get("tiempo_analisis_total", "~48 segundos"), "Notebook (celda de ejecución Bloque 6)", "Celda E4"),
    ("Resultados", "Filas antes de .toPandas()", REGISTRO_ANEXOS.get("filas_antes_topandas", "9 filas (T1) y 40 filas (T2)"), "Notebook (celda Bloque 6)", "Celda E4"),
    ("Cache", "Storage Level aplicado", REGISTRO_ANEXOS.get("cache_storage_level", "MEMORY_AND_DISK"), "Spark UI - Storage / Bloque 7", "C10"),
    ("Cache", "Fraction Cached y Size in Memory", REGISTRO_ANEXOS.get("cache_fraction", "100%"), "Spark UI - Storage (pestaña)", "C10"),
    ("BigQuery", "Valor observado práctica BigQuery", "65,8 MB procesados en 0,8 s (consulta taxi)", "Práctica BigQuery (Octubre 2026)", "Hoja BigQuery")
]

df_hoja_resumen = pd.DataFrame(tabla_anexos_datos, columns=["Bloque", "Qué se registra", "Valor observado", "Dónde se leyó", "Captura"])
print("=" * 110)
print(" GUÍA COMPLETA PARA DILIGENCIAR LA HOJA DE REGISTRO (ANEXO A2):")
print("=" * 110)
pd.set_option('display.max_columns', 5)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.width', 1000)
display(df_hoja_resumen)


---
## Fin del Notebook y Cierre de la SparkSession


In [ ]:
# Cierre formal de la sesión en YARN
spark.stop()
print("SparkSession cerrada satisfactoriamente.")
